# Question-2: CycleGAN for Person Face Sketches
Unpaired photo ↔ sketch translation (Zhu et al., 2017), trained end-to-end on
[Person Face Sketches](https://www.kaggle.com/datasets/almightyj/person-face-sketches).

**Runtime → Change runtime type → GPU.** Checkpoints go to Google Drive. After a disconnect, run the cells again from the top and training picks up from the last saved epoch.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 2. Drive + code
Upload this `q2/` folder to `MyDrive/as2/q2` first. If you use a different location, change `CODE_DIR`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CODE_DIR = '/content/drive/MyDrive/as2/q2'
CKPT = '/content/drive/MyDrive/cyclegan_ckpt'
%cd $CODE_DIR
!pip -q install -r requirements.txt

## 3. Dataset
`kagglehub` asks for your Kaggle credentials if it needs them. The second command prints how many photos and sketches the loader found. If it picked the wrong folders, set `PHOTO_DIR`/`SKETCH_DIR` below.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download("almightyj/person-face-sketches")
print(DATA)
!find "$DATA" -maxdepth 3 -type d | head -20
!python dataset.py --data_dir "$DATA"

In [ ]:
import os
# Train on train/ when the dataset has splits; test images must never be trained on.
ROOT = next((f"{DATA}/{s}" for s in ("train", "Train") if os.path.isdir(f"{DATA}/{s}")), DATA)
TEST_ROOT = next((f"{DATA}/{s}" for s in ("test", "Test", "val", "Val") if os.path.isdir(f"{DATA}/{s}")), None)
print("train root:", ROOT, "| test root:", TEST_ROOT)
# Leave empty to auto-split by folder name, or set explicit training folders, e.g. f"{ROOT}/photos"
PHOTO_DIR, SKETCH_DIR = "", ""
DATA_FLAGS = f'--data_dir "{ROOT}"' + (f' --photo_dir "{PHOTO_DIR}" --sketch_dir "{SKETCH_DIR}"' if PHOTO_DIR and SKETCH_DIR else "")
!python dataset.py $DATA_FLAGS

## 4. Hyperparameter tuning (short runs)
Each setting below trains for 3 epochs at 128 px on 1,000 images per domain, with its own checkpoint dir. We compare
the cycle-consistency loss (lower means better reconstruction) and look at the sample grids. GAN losses alone don't say much about quality.

Note that batch size 4 makes 4× fewer updates in 3 epochs than batch size 1, so its loss is a little higher for that reason alone. Judge it partly on the sample grids.

In [ ]:
TUNE = {
    "lr2e-4_bs1": "--lr 2e-4 --batch_size 1",
    "lr1e-4_bs1": "--lr 1e-4 --batch_size 1",
    "lr2e-4_bs4": "--lr 2e-4 --batch_size 4",
}
for name, flags in TUNE.items():
    !python train.py $DATA_FLAGS --checkpoint_dir "$CKPT/tune_$name" {flags} \
        --epochs 3 --image_size 128 --n_blocks 6 --max_samples 1000 --snapshot_every 0 --amp --log_every 200

In [ ]:
import pandas as pd
rows = []
for name in TUNE:
    h = pd.read_csv(f"{CKPT}/tune_{name}/history.csv")
    rows.append({"run": name, **h.iloc[-1][["loss_G", "loss_D", "loss_cycle", "loss_idt", "seconds"]].to_dict()})
pd.DataFrame(rows).sort_values("loss_cycle")

In [ ]:
from IPython.display import Image, display
for name in TUNE:
    print(name); display(Image(f"{CKPT}/tune_{name}/samples/epoch_003.png"))

## 5. Full training
Put the winning `--lr`/`--batch_size` in `BEST`.

**Timing:** on a T4 at 256 px, one epoch over 3,000 images per domain takes roughly 12–18 minutes, so 30 epochs is about 8 hours spread over a few sessions.

**Resuming:** if Colab disconnects, re-run this cell (and the cells above it) and training continues from `latest.pth`.

**Drive space:** `latest.pth` is about 340 MB and every 5th epoch adds a 91 MB snapshot. Empty Drive's Trash now and then, because overwritten files still count toward your quota.

**Faster option:** `--image_size 128 --n_blocks 6`, which is about 4× quicker.

In [ ]:
BEST = "--lr 2e-4 --batch_size 1"
!python train.py $DATA_FLAGS --checkpoint_dir "$CKPT/main" {BEST} --epochs 30 --max_samples 3000 --image_size 256 --snapshot_every 5 --amp --num_workers 2

## 6. Monitor training

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display
h = pd.read_csv(f"{CKPT}/main/history.csv")
h.plot(x="epoch", y=["loss_G", "loss_D", "loss_cycle", "loss_idt"], figsize=(9, 4), title="CycleGAN losses")
display(h.tail())
latest = sorted(glob.glob(f"{CKPT}/main/samples/epoch_*.png"))[-1]
print(latest, "(rows: photo, ->sketch, sketch, ->photo)"); display(Image(latest))

## 7. Test both directions
Uses the auto-detector: a photo becomes a sketch and a sketch becomes a photo.

In [ ]:
from IPython.display import Image, display
from dataset import resolve_domains
if TEST_ROOT:
    test_photos, test_sketches = resolve_domains(TEST_ROOT)
else:
    print("WARNING: no test/ split found; these images may have been seen in training")
    test_photos, test_sketches = resolve_domains(ROOT, PHOTO_DIR or None, SKETCH_DIR or None)
for i, src in enumerate([test_photos[0], test_sketches[0]]):
    !python inference.py --checkpoint "$CKPT/main/latest.pth" --input "{src}" --output /content/out_{i}.png
    display(Image(str(src), width=256), Image(f"/content/out_{i}.png", width=256))

### Sketch/photo auto-detection accuracy
The UI picks the direction with `inference.is_sketch`. This measures how often it is right on held-out images. Report the number.

In [ ]:
import random
from PIL import Image as PILImage
from inference import is_sketch
rng = random.Random(0)
P, S = rng.sample(test_photos, min(500, len(test_photos))), rng.sample(test_sketches, min(500, len(test_sketches)))
photo_ok = sum(not is_sketch(PILImage.open(p)) for p in P)
sketch_ok = sum(is_sketch(PILImage.open(p)) for p in S)
print(f"photos correctly detected:   {photo_ok}/{len(P)}")
print(f"sketches correctly detected: {sketch_ok}/{len(S)}")
print(f"overall accuracy: {(photo_ok + sketch_ok) / (len(P) + len(S)):.1%}")

## 8. Web UI
Starts Flask in the background and prints the proxied HTTPS URL. Open it in a new tab: upload, camera and live mode all work there.

In [ ]:
!pkill -f "python app.py" || true
!nohup python app.py --checkpoint "$CKPT/main/latest.pth" --port 5000 > app.log 2>&1 &
import time; time.sleep(5); print(open("app.log").read()[-500:])
from google.colab.output import eval_js; print(eval_js("google.colab.kernel.proxyPort(5000)"))

## 9. Before closing the session
Flush Drive so the last checkpoint is fully written.

In [ ]:
from google.colab import drive; drive.flush_and_unmount()